# Short-Stay Listings — filtered & feature-selected from `detailed_listings.csv`

**Sources:**
- `analysis/short_stay_nights.csv` — short-stay calendar rows (`minimum_nights < 29`), produced by `short_stay_nights.ipynb`
- `raw data/detailed_listings.csv`

Kept separate from `short_stay_nights.ipynb` on purpose. This notebook:
1. Filters `detailed_listings.csv` down to the short-stay listing IDs
2. Selects the columns we've decided to keep, split into:
   - **Text** — for embedding later (semantic search: "quiet", "under $200", etc.)
   - **Structured** — dual-purpose: filtering/ranking for the model AND shown on the page
   - **Host profile / display** — needed for the web UI, not fed to the model
3. Drops columns that are either pure scraping metadata, redundant, or **100% empty**
   in this dataset (verified below — no point keeping a column with zero data)

Everything downstream reuses the same `short_stay_listings` DataFrame — no
re-filtering. Display only — **nothing is saved to disk.**

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Get the short-stay listing IDs

In [2]:
short_stay_nights = pd.read_csv("short_stay_nights.csv", usecols=["listing_id"])
short_stay_listing_ids = short_stay_nights["listing_id"].unique()

print(f"Unique short-stay listing IDs: {len(short_stay_listing_ids):,}")

Unique short-stay listing IDs: 1,633


## 2. Filter `detailed_listings.csv`

In [3]:
import re

# id-like columns are 19-digit numbers. Read them as strings explicitly —
# left to default inference, pandas upcasts any column with nulls (like
# host_profile_id, ~4.7% null) to float64, which can't represent 19-digit
# integers exactly and silently corrupts them (e.g. into scientific
# notation). id/host_id have no nulls so they're unaffected either way, but
# stay strings here defensively since they're identifiers, not numbers to
# do math on.
ID_COLUMNS = {"id": str, "host_id": str, "host_profile_id": str}

detailed_listings = pd.read_csv("../raw data/detailed_listings.csv", dtype=ID_COLUMNS)

short_stay_listings = detailed_listings[detailed_listings["id"].isin(short_stay_listing_ids.astype(str))]

print(f"detailed_listings.csv total listings:  {len(detailed_listings):,}")
print(f"Short-stay listing IDs (from calendar): {len(short_stay_listing_ids):,}")
print(f"Matched rows in detailed_listings.csv:  {len(short_stay_listings):,}")
print(f"Short-stay IDs with NO match in detailed_listings.csv: "
      f"{len(set(short_stay_listing_ids.astype(str)) - set(detailed_listings['id'])):,}")

short_stay_listings.head()

detailed_listings.csv total listings:  3,097
Short-stay listing IDs (from calendar): 1,633
Matched rows in detailed_listings.csv:  1,483
Short-stay IDs with NO match in detailed_listings.csv: 150


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
48,4499317,https://www.airbnb.com/rooms/4499317,20260629151954,2026-06-29,city scrape,Heritage Collection on Seah Premium Loft,"Our Premium Loft is a chic, cosy apartment whi...",NaN,https://a0.muscache.com/pictures/miso/Hosting-...,23336011,https://www.airbnb.com/users/show/23336011,1465271884616547411,https://www.airbnb.com/users/profile/146527188...,Heritage,NaN,11.00,7.00,11.00,7.00,Singapore,Hi Everyone! \n\nWe are the management team be...,NaN,NaN,NaN,t,NaN,https://a0.muscache.com/im/pictures/user/User-...,NaN,82.00,NaN,NaN,t,t,NaN,Downtown Core,Central Region,1.30,103.85,Room in aparthotel,Entire home/apt,2,1.00,1 bath,NaN,1.00,"[""Wifi"", ""Bidet"", ""Self check-in"", ""Smart lock...",$114.00,2026-07-01,2026-07-03,228.00,114.00,"{""quote"": {""taxes"": null, ""currency"": ""SGD"", ""...",2,365,2,2,89,365,2.00,168.40,NaN,t,26,54,84,359,2026-06-29,298,19,0,180,18,114,"12,996.00",2014-11-15,2026-05-23,4.83,4.83,4.81,4.78,4.89,4.92,4.73,S0448,NaN,71,47,24,0,2.11
49,4541183,https://www.airbnb.com/rooms/4541183,20260629151954,2026-06-29,city scrape,Boat Quay Studio Premium River View (South bri...,Our Studio Premium Apartments are ideal for co...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,23336011,https://www.airbnb.com/users/show/23336011,1465271884616547411,https://www.airbnb.com/users/profile/146527188...,Heritage,NaN,11.00,7.00,11.00,7.00,Singapore,Hi Everyone! \n\nWe are the management team be...,NaN,NaN,NaN,t,NaN,https://a0.muscache.com/im/pictures/user/User-...,NaN,82.00,NaN,NaN,t,t,NaN,Singapore River,Central Region,1.29,103.85,Room in aparthotel,Entire home/apt,2,NaN,1 bath,NaN,1.00,"[""Wifi"", ""Bidet"", ""Iron"", ""Refrigerator"", ""Sel...",$159.00,2026-07-01,2026-07-07,954.00,159.00,"{""quote"": {""taxes"": null, ""currency"": ""SGD"", ""...",6,365,6,6,89,365,6.00,126.10,NaN,t,28,58,88,361,2026-06-29,376,40,7,182,24,255,"40,545.00",2014-11-29,2026-06-26,4.78,4.80,4.83,4.76,4.89,4.86,4.69,Authorised Serviced Apartment,NaN,71,47,24,0,2.67
57,4973227,https://www.airbnb.com/rooms/4973227,20260629151954,2026-06-29,city scrape,Boat Quay Studio Apartment (South bridge Wing),Our Studio Apartments are ideal for couples tr...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,23336011,https://www.airbnb.com/users/show/23336011,1465271884616547411,https://www.airbnb.com/users

## 3. Confirm which columns are actually empty

Before finalizing the drop list, verify these columns really have zero data
in `short_stay_listings` (not just the full `detailed_listings.csv`).

In [4]:
fully_empty = [
    "instant_bookable", "host_response_time", "host_response_rate", "host_acceptance_rate",
    "host_thumbnail_url", "host_since", "host_neighbourhood", "host_total_listings_count",
    "host_verifications", "neighborhood_overview", "neighbourhood", "calendar_updated",
]

null_pct = (short_stay_listings[fully_empty].isna().mean() * 100).round(1)
print(null_pct.to_string())
assert (null_pct == 100.0).all(), "Expected all of these to be 100% null — double-check before dropping."
print("\nConfirmed — all 12 columns are 100% empty. Safe to drop.")

instant_bookable            100.00
host_response_time          100.00
host_response_rate          100.00
host_acceptance_rate        100.00
host_thumbnail_url          100.00
host_since                  100.00
host_neighbourhood          100.00
host_total_listings_count   100.00
host_verifications          100.00
neighborhood_overview       100.00
neighbourhood               100.00
calendar_updated            100.00

Confirmed — all 12 columns are 100% empty. Safe to drop.


## 4. Select the final feature set

- **Text** — for embedding later
- **Structured** — filtering/ranking for the model, also shown on the page
- **Host profile / display** — for the web UI only
- **Key** — join keys, not a "feature" but needed either way

In [5]:
TEXT_FEATURES = [
    "name", "description", "amenities", "property_type", "room_type",
]

STRUCTURED_FEATURES = [
    "price",
    "neighbourhood_cleansed", "neighbourhood_group_cleansed",
    "latitude", "longitude",
    "accommodates", "bedrooms", "beds", "bathrooms_text",   # bathrooms_text replaces numeric `bathrooms` (62.8% null vs 0.3%)
    "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication", "review_scores_location", "review_scores_value",
    "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d", "number_of_reviews_ly",
    "reviews_per_month", "first_review", "last_review",
    "minimum_nights", "maximum_nights",
    "availability_30", "availability_60", "availability_90", "availability_365", "has_availability",
    "host_is_superhost",
]

HOST_DISPLAY_FEATURES = [
    "picture_url", "listing_url",
    "host_name", "host_url", "host_picture_url", "host_profile_id", "host_profile_url",
    "host_about", "host_location", "host_identity_verified", "host_has_profile_pic",
    "host_listings_count", "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes", "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",
    "hosts_time_as_user_years", "hosts_time_as_user_months",
    "hosts_time_as_host_years", "hosts_time_as_host_months",
    "license",
]

KEY_COLUMNS = ["id", "host_id"]

SELECTED_COLUMNS = KEY_COLUMNS + TEXT_FEATURES + STRUCTURED_FEATURES + HOST_DISPLAY_FEATURES
print(f"Total columns selected: {len(SELECTED_COLUMNS)} / {short_stay_listings.shape[1]}")
print(f"  Text:          {len(TEXT_FEATURES)}")
print(f"  Structured:    {len(STRUCTURED_FEATURES)}")
print(f"  Host/display:  {len(HOST_DISPLAY_FEATURES)}")
print(f"  Key:           {len(KEY_COLUMNS)}")

Total columns selected: 59 / 90
  Text:          5
  Structured:    31
  Host/display:  21
  Key:           2


In [6]:
features_df = short_stay_listings[SELECTED_COLUMNS]
features_df.head()

,id,host_id,name,description,amenities,property_type,room_type,price,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,accommodates,bedrooms,beds,bathrooms_text,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,number_of_reviews_ly,reviews_per_month,first_review,last_review,minimum_nights,maximum_nights,availability_30,availability_60,availability_90,availability_365,has_availability,host_is_superhost,picture_url,listing_url,host_name,host_url,host_picture_url,host_profile_id,host_profile_url,host_about,host_location,host_identity_verified,host_has_profile_pic,host_listings_count,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,license
48,4499317,23336011,Heritage Collection on Seah Premium Loft,"Our Premium Loft is a chic, cosy apartment whi...","[""Wifi"", ""Bidet"", ""Self check-in"", ""Smart lock...",Room in aparthotel,Entire home/apt,$114.00,Downtown Core,Central Region,1.30,103.85,2,NaN,1.00,1 bath,4.83,4.83,4.81,4.78,4.89,4.92,4.73,298,19,0,18,2.11,2014-11-15,2026-05-23,2,365,26,54,84,359,t,t,https://a0.muscache.com/pictures/miso/Hosting-...,https://www.airbnb.com/rooms/4499317,Heritage,https://www.airbnb.com/users/show/23336011,https://a0.muscache.com/im/pictures/user/User-...,1465271884616547411,https://www.airbnb.com/users/profile/146527188...,Hi Everyone! \n\nWe are the management team be...,Singapore,t,t,82.00,71,47,24,0,11.00,7.00,11.00,7.00,S0448
49,4541183,23336011,Boat Quay Studio Premium River View (South bri...,Our Studio Premium Apartments are ideal for co...,"[""Wifi"", ""Bidet"", ""Iron"", ""Refrigerator"", ""Sel...",Room in aparthotel,Entire home/apt,$159.00,Singapore River,Central Region,1.29,103.85,2,NaN,1.00,1 bath,4.78,4.80,4.83,4.76,4.89,4.86,4.69,376,40,7,24,2.67,2014-11-29,2026-06-26,6,365,28,58,88,361,t,t,https://a0.muscache.com/pictures/hosting/Hosti...,https://www.airbnb.com/rooms/4541183,Heritage,https://www.airbnb.com/users/show/23336011,https://a0.muscache.com/im/pictures/user/User-...,1465271884616547411,https://www.airbnb.com/users/profile/146527188...,Hi Everyone! \n\nWe are the management team be...,Singapore,t,t,82.00,71,47,24,0,11.00,7.00,11.00,7.00,Authorised Serviced Apartment
57,4973227,23336011,Boat Quay Studio Apartment (South bridge Wing),Our Studio Apartments are ideal for couples tr...,"[""Wifi"", ""Bidet"", ""Iron"", ""Refrigerator"", ""Sel...",Room in aparthotel,Entire home/apt,$174.33,Singapore River,Central Region,1.29,103.85,2,NaN,1.00,1 bath,4.71,4.73,4.77,4.64,4.86,4.88,4.62,184,4,0,7,1.34,2015-03-06,2026-02-07,6,365,25,55,85,358,t,t,https://a0.muscache.com/pictures/hosting/Hosti...,https://www.airbnb.com/rooms/4973227,Heritage,https://www.airbnb.com/users/show/23336011,https://a0.muscache.com/im/pictures/user/User-...,1465271884616547411,https://www.airbnb.com/users/profile/146527188...,Hi Everyone! \n\nWe are the management team be...,Singapore,t,t,82.00,71,47,24,0,11.00,7.00,11.00,7.00,Authorised Serviced Apartment
60,5052014,13468566,Modern Serviced Studio in CBD w MRT,We offer a modern design ensuite studio with h...,"[""Wifi"", ""Refrigerator"", ""Hot water kettle"", ""...",Entire serviced apartment,Entire home/apt,$154.17,Outram,Central Region,1.27,103.84,2,NaN,1.00,1 bath,4.69,4.63,4.88,4.69,4.75,4.81,4.63,16,2,0,1,0.12,2015-03-21,2026-02-18,6,1125,27,44,72,347,t,f,https://a0.muscache.com/pictures/66917563/13b9...,https://www.airbnb.com/rooms/5052014,Andy,https://www.airbnb.com/users/show/13468566,https://a0.muscache.com/im/users/13468566/prof...,1462586490802398761,https://www.airbnb.com/users/profile/146258649...,Andy has made it his personal mission to hu

## 4b. Fix data types & flag review availability

Two columns came through as raw strings instead of usable types:
- `price` — e.g. `"$114.00"` (dollar sign, thousands comma) instead of a number
- `amenities` — a JSON-encoded list as one string instead of clean text

And for the review columns (~36% null): those nulls are meaningful — the
listing genuinely has no reviews yet, not missing data. Per plan, we don't
impute them here — just add a `has_reviews` flag so a model can tell "new,
no track record" apart from "established, average rating." No one-hot
encoding for categoricals at this stage either (`room_type`,
`neighbourhood_cleansed`, etc. stay as plain text columns — embedding
layers, not one-hot, come later at model-build time).

In [7]:
import json

# --- price: "$114.00" / "$1,274.00" -> 114.0 / 1274.0 (float) ---
features_df["price"] = (
    features_df["price"]
        .str.replace(r"[$,]", "", regex=True)
        .astype(float)
)

# --- amenities: JSON-encoded list string -> clean comma-separated text ---
def clean_amenities(raw):
    if pd.isna(raw):
        return raw
    try:
        items = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return raw
    return ", ".join(items)

features_df["amenities"] = features_df["amenities"].apply(clean_amenities)

# --- has_reviews: flag, not impute -- some listings genuinely have no reviews yet ---
features_df["has_reviews"] = features_df["number_of_reviews"].fillna(0).gt(0).astype(int)

print("price dtype:", features_df["price"].dtype)
print("\nhas_reviews value counts:")
print(features_df["has_reviews"].value_counts())
print("\nsample:")
features_df[["price", "amenities", "has_reviews"]].head(3)

price dtype: float64

has_reviews value counts:
has_reviews
1    950
0    533
Name: count, dtype: int64

sample:


,price,amenities,has_reviews
48,114.00,"Wifi, Bidet, Self check-in, Smart lock, Hot wa...",1
49,159.00,"Wifi, Bidet, Iron, Refrigerator, Self check-in...",1
57,174.33,"Wifi, Bidet, Iron, Refrigerator, Self check-in...",1


In [8]:
import re

# `description` carries raw HTML (<br />, etc.) in ~41% of listings — same
# artifact we found and stripped from review comments earlier. Clean it here
# too: it's both displayed on the web app and fed to the text embedding later.
def strip_html(text):
    if pd.isna(text):
        return text
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

before = features_df["description"].fillna("").str.contains(r"<[a-zA-Z/][^>]*>", regex=True).sum()
features_df["description"] = features_df["description"].apply(strip_html)
after = features_df["description"].fillna("").str.contains(r"<[a-zA-Z/][^>]*>", regex=True).sum()

print(f"description: {before} rows had HTML tags before cleaning, {after} after")

description: 604 rows had HTML tags before cleaning, 0 after


## 5. Null check on the final selection

In [9]:
null_pct_final = (features_df.isna().mean() * 100).round(1).sort_values(ascending=False)
null_pct_final[null_pct_final > 0]

host_location                 49.00
host_about                    43.50
review_scores_rating          35.90
last_review                   35.90
first_review                  35.90
reviews_per_month             35.90
review_scores_value           35.90
review_scores_location        35.90
review_scores_communication   35.90
review_scores_cleanliness     35.90
review_scores_accuracy        35.90
review_scores_checkin         35.90
bedrooms                      21.40
beds                          14.70
description                    9.40
license                        8.30
price                          5.90
host_profile_url               4.70
host_listings_count            4.00
hosts_time_as_host_months      4.00
hosts_time_as_host_years       4.00
host_identity_verified         4.00
host_has_profile_pic           4.00
host_picture_url               4.00
hosts_time_as_user_months      4.00
hosts_time_as_user_years       4.00
host_profile_id                4.00
host_is_superhost           

## Dropped columns and why

- **100% empty in this dataset** (confirmed in section 3): `instant_bookable`,
  `host_response_time`, `host_response_rate`, `host_acceptance_rate`,
  `host_thumbnail_url`, `host_since`, `host_neighbourhood`,
  `host_total_listings_count`, `host_verifications`, `neighborhood_overview`,
  `neighbourhood` (raw — replaced by `neighbourhood_cleansed`/`neighbourhood_group_cleansed`),
  `calendar_updated`
- **Pure scraping metadata**: `scrape_id`, `last_scraped`, `source`, `calendar_last_scraped`
- **Redundant with a column we kept**: `bathrooms` (→ `bathrooms_text`),
  `minimum_minimum_nights`, `maximum_minimum_nights`, `minimum_maximum_nights`,
  `maximum_maximum_nights`, `minimum_nights_avg_ntm`, `maximum_nights_avg_ntm`
  (all redundant with `minimum_nights`/`maximum_nights`)
- **A date-specific price quote, not a stable price**: `price_quote_checkin_date`,
  `price_quote_checkout_date`, `price_quote_total_price`, `price_quote_price_per_night`,
  `price_quote_raw` (`price` already covers this)
- **Host/business analytics, not guest-facing**: `estimated_occupancy_l365d`,
  `estimated_revenue_l365d`

## 6. Save the selected features

Save `features_df` (1,483 short-stay listings × 59 columns — display data and
deep-learning features together) to `analysis/short_stay_listing_features.csv`.
Feature engineering (cleaning text, imputing nulls, encoding, embeddings)
happens later — this is the selected-but-unprocessed feature set.

In [10]:
features_df.to_csv("short_stay_listing_features.csv", index=False)
print(f"Saved {len(features_df):,} rows x {features_df.shape[1]} columns to "
      f"analysis/short_stay_listing_features.csv")

Saved 1,483 rows x 60 columns to analysis/short_stay_listing_features.csv


## Note for later — `reviews.csv` review dates aren't in `detailed_listings.csv`

`detailed_listings.csv` only carries aggregate review fields (`first_review`,
`last_review`, `number_of_reviews`, ...) — it has **no per-review date**. The
individual review dates needed for the planned deep-learning work live only
in `reviews.csv` (`listing_id`, `date`, ...) and would need to be joined in
separately (on `listing_id`) when that work starts. Not handled in this
notebook — flagged here for later.
